# 05 — Recherche lexicale TF-IDF

**Objectif :** retrouver les produits pertinents avec des n-grams de mots et de caractères.

**Entrées :** produits nettoyés et validation du notebook 03.  
**Sorties :** vectorizers, matrices et métriques TF-IDF.  
**Dépendances :** notebooks 02 et 03.  
**Temps estimé :** moins de deux minutes.  
**Ressources :** CPU et mémoire vive.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.feature_extraction.text import TfidfVectorizer


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metrics import evaluate_rankings

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
TOP_K = int(CONFIG["project"]["top_k"])
PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
ARTIFACTS_DIR = ROOT / CONFIG["paths"]["artifacts"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

products = pd.read_csv(PROCESSED_DIR / "products_clean.csv", dtype={"sku": str}).fillna("")
validation = pd.read_csv(PROCESSED_DIR / "validation_clicks.csv", dtype={"sku": str}).fillna("")
assert products["sku"].is_unique

## Index mots et caractères

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    norm="l2",
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
    norm="l2",
)
word_matrix = word_vectorizer.fit_transform(products["document"])
char_matrix = char_vectorizer.fit_transform(products["document"])

print("Matrice mots :", word_matrix.shape)
print("Matrice caractères :", char_matrix.shape)


def lexical_ranking(query: str, k: int = TOP_K) -> list[str]:
    word_scores = (word_vectorizer.transform([query]) @ word_matrix.T).toarray()[0]
    char_scores = (char_vectorizer.transform([query]) @ char_matrix.T).toarray()[0]
    scores = 0.5 * word_scores + 0.5 * char_scores
    order = np.argsort(-scores, kind="stable")
    return products.iloc[order[:k]]["sku"].astype(str).tolist()


for query in ("call duty", "mine craft", "fast car game"):
    print(query, "→", lexical_ranking(query))

## Évaluation

In [ ]:
actual_by_query = validation.groupby("query_key")["sku"].agg(lambda values: set(map(str, values)))
query_text_by_key = validation.groupby("query_key")["query_text"].first()
predictions = [lexical_ranking(query_text_by_key.loc[key]) for key in actual_by_query.index]
assert all(len(items) == len(set(items)) == TOP_K for items in predictions)

metrics = evaluate_rankings(actual_by_query.tolist(), predictions, TOP_K)
report = {
    "model": "tfidf_word_char",
    "word_vocabulary": len(word_vectorizer.vocabulary_),
    "char_vocabulary": len(char_vectorizer.vocabulary_),
    **metrics,
}

joblib.dump(word_vectorizer, ARTIFACTS_DIR / "word_tfidf.joblib")
joblib.dump(char_vectorizer, ARTIFACTS_DIR / "char_tfidf.joblib")
joblib.dump(word_matrix, ARTIFACTS_DIR / "word_product_matrix.joblib")
joblib.dump(char_matrix, ARTIFACTS_DIR / "char_product_matrix.joblib")
(REPORTS_DIR / "metrics_tfidf.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Conclusion

Les caractères rendent la recherche tolérante aux espaces et fautes légères, tandis que les mots
favorisent les correspondances exactes de titres et catégories.